# Gold Standard Sampling
Fetching judgments outside LIDO for manual annotation, from Rechtspraak API.

**Required files in the same directory:**
- `eclis.csv.gz` — all ECLIs from rechtspraak.nl
- `links.csv.gz` — LIDO annotations

**Output:**
- `gold_standard_sample.csv.gz` — 1000 judgments with text


In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import random
import time

CONTENT_URL = "https://data.rechtspraak.nl/uitspraken/content"
SEARCH_URL  = "https://data.rechtspraak.nl/uitspraken/zoeken"
NS = {"atom": "http://www.w3.org/2005/Atom"}

# Creating sample from local dataset

In [ ]:
# LIDO ECLIs
df_links = pd.read_csv("data/links.csv.gz", compression="gzip", usecols=["ecli"])
lido_eclis = set(df_links["ecli"].dropna().unique())
print(f"ECLIs in LIDO: {len(lido_eclis):,}")

# Rechtspraak.nl ECLIs
df_eclis = pd.read_csv("data/eclis.csv.gz", compression="gzip")
print(f"ECLIs on rechtspraak.nl: {len(df_eclis):,}")

candidates = df_eclis[~df_eclis["ecli"].isin(lido_eclis)].copy()
candidates_set = set(candidates["ecli"].tolist())

print(f"Candidates outside LIDO:      {len(candidates):,}")


In [ ]:
random.seed(42)
N_TOTAL = 1000

sample_eclis = random.sample(list(candidates_set), N_TOTAL)
df_sample = pd.DataFrame({"ecli": sample_eclis})

print(f"Randomly sampled: {len(df_sample)} judgments")
df_sample.head()


In [6]:
def fetch_text(ecli):
    """Fetches plain text, date, and rechtsgebied via the content API."""
    try:
        r = requests.get(
            f"https://data.rechtspraak.nl/uitspraken/content?id={ecli}",
            timeout=10
        )

        if r.status_code == 429:
            print("  Rate limit hit, waiting 30s...")
            time.sleep(30)
            r = requests.get(
                f"https://data.rechtspraak.nl/uitspraken/content?id={ecli}",
                timeout=10
            )

        if r.status_code != 200:
            return None, None, None

        root = ET.fromstring(r.content)
        ns = {"rs": "http://www.rechtspraak.nl/schema/rechtspraak-1.0"}

        # Judgment text — only the uitspraak/conclusie body
        def get_text(element):
            text = element.text or ""
            for child in element:
                text += get_text(child)
                text += child.tail or ""
            return text

        uitspraak = root.find(".//rs:uitspraak", ns)
        if uitspraak is None:
            uitspraak = root.find(".//rs:conclusie", ns)

        if uitspraak is None or len(get_text(uitspraak).strip()) < 200:
            return None, None, None

        text = get_text(uitspraak).strip()

        # Date
        date = None
        for tag in ["{http://purl.org/dc/terms/}date",
                    "{http://www.rechtspraak.nl/schema/rechtspraak-1.0}date"]:
            elem = root.find(f".//{tag}")
            if elem is not None and elem.text:
                date = elem.text.strip()
                break

        # Rechtsgebied
        rechtsgebied = None
        for tag in ["{http://purl.org/dc/terms/}subject",
                    "{http://www.rechtspraak.nl/schema/rechtspraak-1.0}subject"]:
            elem = root.find(f".//{tag}")
            if elem is not None and elem.text:
                rechtsgebied = elem.text.strip()
                break

        return text, date, rechtsgebied

    except Exception:
        return None, None, None

In [7]:
texts, dates, rechtsgebieden = [], [], []
failed = 0

for i, row in df_sample.iterrows():
    text, date, rechtsgebied = fetch_text(row["ecli"])
    texts.append(text)
    dates.append(date)
    rechtsgebieden.append(rechtsgebied)

    if text is None:
        failed += 1

    if (i + 1) % 20 == 0:
        print(f"{i + 1}/{len(df_sample)} fetched ({failed} failed)")

    time.sleep(0.3)

df_sample["text"]             = texts
df_sample["date"]             = dates
df_sample["rechtsgebied"]     = rechtsgebieden

df_sample = df_sample[df_sample["text"].notna()].reset_index(drop=True)
print(f"\nRemaining after filtering: {len(df_sample)}")
df_sample[["ecli", "date", "rechtsgebied"]].head(10)

20/1000 fetched (0 failed)
40/1000 fetched (0 failed)
60/1000 fetched (0 failed)
80/1000 fetched (0 failed)
100/1000 fetched (0 failed)
120/1000 fetched (0 failed)
140/1000 fetched (0 failed)
160/1000 fetched (0 failed)
180/1000 fetched (0 failed)
200/1000 fetched (0 failed)
220/1000 fetched (0 failed)
240/1000 fetched (0 failed)
260/1000 fetched (0 failed)
280/1000 fetched (0 failed)
300/1000 fetched (0 failed)
320/1000 fetched (0 failed)
340/1000 fetched (0 failed)
360/1000 fetched (0 failed)
380/1000 fetched (0 failed)
400/1000 fetched (0 failed)
420/1000 fetched (0 failed)
440/1000 fetched (0 failed)
460/1000 fetched (0 failed)
480/1000 fetched (0 failed)
500/1000 fetched (0 failed)
520/1000 fetched (0 failed)
540/1000 fetched (0 failed)
560/1000 fetched (0 failed)
580/1000 fetched (0 failed)
600/1000 fetched (0 failed)
620/1000 fetched (0 failed)
640/1000 fetched (0 failed)
660/1000 fetched (0 failed)
680/1000 fetched (0 failed)
700/1000 fetched (0 failed)
720/1000 fetched (0 fail

,ecli,date,rechtsgebied
0,ECLI:NL:RBDHA:2025:8235,2025-04-10,Bestuursrecht; Vreemdelingenrecht
1,ECLI:NL:OGAACMB:2017:68,2017-08-28,Bestuursrecht; Ambtenarenrecht
2,ECLI:NL:RBSGR:2012:BW2477,2012-04-16,Civiel recht
3,ECLI:NL:RBDHA:2021:16863,2021-11-23,Bestuursrecht; Vreemdelingenrecht
4,ECLI:NL:RBDHA:2025:4857,2025-03-25,Bestuursrecht; Vreemdelingenrecht
5,ECLI:NL:RBNNE:2024:4711,2024-11-14,Bestuursrecht; Belastingrecht
6,ECLI:NL:RVS:2025:2901,2025-07-01,Bestuursrecht; Vreemdelingenrecht
7,ECLI:NL:RBARN:2006:3841,2006-06-21,Civiel recht
8,ECLI:NL:RBDHA:2025:13255,2025-07-18,Bestuursrecht; Vreemdelingenrecht
9,ECLI:NL:CBB:2025:206,2025-02-27,Bestuursrecht


In [ ]:
# Save sample permanently
df_sample.to_csv("gold_standard_sample.csv.gz", index=False, compression="gzip")
print(f"Saved: {len(df_sample)} judgments")

# Small EDA

In [8]:
df_sample.head(10)

,ecli,text,date,rechtsgebied
0,ECLI:NL:RBDHA:2025:8235,RECHTBANK DEN HAAG\n \n \n Bestuursrecht\...,2025-04-10,Bestuursrecht; Vreemdelingenrecht
1,ECLI:NL:OGAACMB:2017:68,Uitspraak van 28 augustus 2017\t\t\t\t\t\n ...,2017-08-28,Bestuursrecht; Ambtenarenrecht
2,ECLI:NL:RBSGR:2012:BW2477,RECHTBANK 'S-GRAVENHAGE\n \n Sector civi...,2012-04-16,Civiel recht
3,ECLI:NL:RBDHA:2021:16863,RECHTBANK DEN HAAG\n \n \n Zittings...,2021-11-23,Bestuursrecht; Vreemdelingenrecht
4,ECLI:NL:RBDHA:2025:4857,RECHTBANK DEN HAAG\n \n \n Zittings...,2025-03-25,Bestuursrecht; Vreemdelingenrecht
5,ECLI:NL:RBNNE:2024:4711,RECHTBANK NOORD-NEDERLAND\n \n \n Zitting...,2024-11-14,Bestuursrecht; Belastingrecht
6,ECLI:NL:RVS:2025:2901,BRS.25.000062\n ECLI:NL:RVS:2025:2901\n ...,2025-07-01,Bestuursrecht; Vreemdelingenrecht
7,ECLI:NL:RBARN:2006:3841,vonni\n s\n \n \n \n RE...,2006-06-21,Civiel recht
8,ECLI:NL:RBDHA:2025:13255,RECHTBANK DEN HAAG\n \n \n Zittings...,2025-07-18,Bestuursrecht; Vreemdelingenrecht
9,ECLI:NL:CBB:2025:206,proces-verbaal uitspraak \n \n \n \n ...,2025-02-27,Bestuursrecht


In [9]:
print("=== Distribution by rechtsgebied ===")
print(df_sample["rechtsgebied"].value_counts(dropna=False).to_string())

print("\n=== Date range ===")
print(f"Earliest: {df_sample['date'].min()}")
print(f"Latest:   {df_sample['date'].max()}")

print("\n=== Text length (characters) ===")
print(df_sample["text"].str.len().describe().round(0))

=== Distribution by rechtsgebied ===
rechtsgebied
Civiel recht                                  192
Strafrecht                                    171
Bestuursrecht; Vreemdelingenrecht             145
Bestuursrecht                                 131
Civiel recht; Personen- en familierecht        89
Bestuursrecht; Belastingrecht                  69
Civiel recht; Verbintenissenrecht              55
Bestuursrecht; Socialezekerheidsrecht          34
Bestuursrecht; Bestuursprocesrecht             22
Bestuursrecht; Omgevingsrecht                  19
Bestuursrecht; Ambtenarenrecht                 10
Civiel recht; Arbeidsrecht                     10
Civiel recht; Ondernemingsrecht                10
Civiel recht; Insolventierecht                 10
Civiel recht; Burgerlijk procesrecht            9
Strafrecht; Europees strafrecht                 7
Strafrecht; Materieel strafrecht                6
Strafrecht; Strafprocesrecht                    5
Bestuursrecht; Bestuursstrafrecht               5


In [10]:
df_sample["text"].str.len().describe()

count      1000.000000
mean      15714.258000
std       21912.980616
min         297.000000
25%        6439.750000
50%       10557.500000
75%       17599.500000
max      454585.000000
Name: text, dtype: float64

In [11]:
# Bekijk de verdeling beter
print("Uitspraken per lengtecat:")
bins = [0, 10000, 30000, 60000, float("inf")]
labels = ["kort (<10k)", "middel (10-30k)", "lang (30-60k)", "zeer lang (>60k)"]
df_sample["lengte_cat"] = pd.cut(df_sample["text"].str.len(), bins=bins, labels=labels)
print(df_sample["lengte_cat"].value_counts().sort_index())

Uitspraken per lengtecat:
lengte_cat
kort (<10k)         470
middel (10-30k)     428
lang (30-60k)        79
zeer lang (>60k)     23
Name: count, dtype: int64
